In [1]:
import requests
import time
import os
import io
import zipfile
from edinet_xbrl.edinet_xbrl_parser import EdinetXbrlParser
import pandas as pd
from datetime import date, timedelta
import logging

import os
from dotenv import load_dotenv
load_dotenv()
EDINET_API_KEY = os.getenv('EDINET_API_KEY')

def get_documents_list(target_date, doc_type='030000'):
    """
    指定した日付にEDINETで開示された書類一覧を取得し、
    指定doc_type(有報)を満たすdoc_idとedinet_codeのリストを返す。
    """
    if isinstance(target_date, date):
        date_str = target_date.strftime("%Y-%m-%d")
    else:
        date_str = target_date
    base_url = "https://disclosure.edinet-fsa.go.jp/api/v2/documents.json"
    params = {'date': date_str, 'type': 2,'Subscription-Key':EDINET_API_KEY}
    r = requests.get(base_url, params=params)
    r.raise_for_status()
    data = r.json()
    results = data.get('results', [])

    docs = []

    #formCode：上場企業を指定しているように見える。
    #docTypeCode：有価証券報告書を指定。大量保有報告書などを指定しない
    for d in results:
        if d.get('formCode') == doc_type and d.get('docTypeCode') == '120':
            docs.append(d)

    return docs

TARGET_DATE = str(date(2024, 12, 20))
DOC_TYPE = '030000'  # 有価証券報告書

d = get_documents_list(TARGET_DATE,DOC_TYPE)

def download_and_parse_xbrl(doc_id):
    base_url = "https://disclosure.edinet-fsa.go.jp/api/v2/documents"
    params = {
        'type': 1,  # ZIP形式
        'Subscription-Key': EDINET_API_KEY
    }

    try:
        r = requests.get(f"{base_url}/{doc_id}", params=params, stream=True)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        logging.error("Failed to download doc_id=%s: %s", doc_id, e)
        return None

    try:
        z = zipfile.ZipFile(io.BytesIO(r.content))
    except zipfile.BadZipFile as e:
        logging.error("Invalid ZIP file for doc_id=%s: %s", doc_id, e)
        return None

    xbrl_files = [f for f in z.namelist() if f.endswith('.xbrl')]
    if not xbrl_files:
        logging.warning("No XBRL file found in the ZIP for doc_id=%s", doc_id)
        return None

    xbrl_path_in_zip = xbrl_files[0]

    # 一時ディレクトリを用意してZIPを解凍
    temp_dir = "tmp_xbrl"
    os.makedirs(temp_dir, exist_ok=True)

    # ZIPの対象ファイルを解凍し、ローカルパスを取得
    z.extract(xbrl_path_in_zip, path=temp_dir)
    xbrl_full_path = os.path.join(temp_dir, xbrl_path_in_zip)

    parser = EdinetXbrlParser()
    # ここで実際にローカルファイルを開く
    parsed_xbrl = parser.parse_file(xbrl_full_path)

    return parsed_xbrl

In [2]:
def get_documents_by_date(target_date, doc_type='030000'):
    """
    指定した日付にEDINETで開示された書類一覧を取得し、
    指定doc_type(有報)を満たすdoc_idとedinet_codeのリストを返す。
    """
    if isinstance(target_date, date):
        date_str = target_date.strftime("%Y-%m-%d")
    else:
        date_str = target_date
    base_url = "https://disclosure.edinet-fsa.go.jp/api/v2/documents.json"
    params = {'date': date_str, 'type': 2,'Subscription-Key':EDINET_API_KEY}
    r = requests.get(base_url, params=params)
    r.raise_for_status()
    data = r.json()
    results = data.get('results', [])

    docs = []


    for d in results:
        if d.get('formCode') == doc_type:
            docs.append(d)

    return docs

In [3]:
# # 設定パラメータ
########################################
# 対象とする日付（有報提出日）
TARGET_DATE = str(date(2024, 12, 20))
DOC_TYPE = '030000'  # 有価証券報告書


In [4]:
d = get_documents_list(TARGET_DATE)

In [5]:
h = download_and_parse_xbrl(d[0]['docID'])

/Users/satoki252595/.pyenv/versions/3.11.7/lib/python3.11/site-packages/xbrl/xbrl.py:26: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  soup = BeautifulSoup(fh, "lxml")


In [6]:
h

In [7]:
#①事業等のリスクをとってみる
key='jpcrp_cor:BusinessRisksTextBlock'
context_ref='FilingDateInstant'
data = h.get_data_by_context_ref(key, context_ref)
text_data = data.get_value()

#②経営方針、経営環境および対処すべき課題等をとってみる
key='jpcrp_cor:BusinessPolicyBusinessEnvironmentIssuesToAddressEtcTextBlock'
context_ref='FilingDateInstant'
data = h.get_data_by_context_ref(key, context_ref)
text_data = data.get_value()

import re
text_data = re.sub('\s','',text_data)
text_data = re.sub('<.*?>','',text_data)

In [8]:
text_data

'１【経営方針、経営環境及び対処すべき課題等】文中の将来に関する事項は、当連結会計年度末現在において、当社グループが判断したものであります。(1)会社の経営の基本方針当社グループの企業理念は、「QualityfortheCustomers=ValuefortheCompany,theEmployees,theSocietyandtheInvestors;EnvironmentfortheSociety=ValuefortheCustomers,theCompany,theEmployeesandtheInvestors」としております。(2)経営上の目標の達成状況を判断するための客観的な指標等当社グループは、長期的な視野に立った企業価値の向上を目指してまいります。当社グループは、財政状態の健全性を示す自己資本比率と収益性を示すROE（株主資本当期純利益率）とのバランスを考え、具体的には、自己資本比率70％以上、ROE15％以上を長期的な経営指標の目標としてまいります。なお、将来に関する事項については達成を保証するものではありません。(3)中長期的な会社の経営戦略当社グループは、創業以来、主に自動車業界を主要顧客とした溶接機器関連事業を中核としてグループの発展を目指してまいりましたが、2000年８月にスピードファム株式会社の株式を100％取得し完全子会社化して以来、溶接機器関連事業と平面研磨装置関連事業という異なる２つの事業に大別される企業集団になりました。そして、2011年10月３日には、各事業の採算性や責任体制の明確化を図るとともに、機動的な対応が可能なグループ運営体制にするため、持株会社体制に移行しました。今後とも、当社グループは、自動車業界とエレクトロニクス業界という二大基幹産業に寄与する企業集団として、グローバルな展開を行い、かつ個々のローカル市場で優位性を確立し、独自の技術を生かした事業の発展を加速させていきたいと考えております。(4)会社の対処すべき課題当社グループの主要顧客は、自動車業界とエレクトロニクス業界であります。自動車業界については、生産コストの削減、新興国を中心とした生産ラインの更新、エコカーの拡充が実施されております。また、自動車需要も新興国経済の発展に伴い、成長が予想されます。エレクトロニクス業界については、短期的な需要変動はあるにしても